In [11]:
import sys
from pathlib import Path
from time import perf_counter

import joblib
import pandas as pd
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
DATA_DIR = ROOT / "datasets" / "final" / "ml"
MODEL_DIR = ROOT / "models" / "randomized_search"
TRAINING_RESULTS_DIR = ROOT / "results" / "model_training" / "randomized_search"
OUTPUT_DIR = ROOT / "results" / "model_evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))
from src.utils.features import split_features

reference_df = pd.read_parquet(DATA_DIR / "cicids2017_train.parquet")
numerical_features, categorical_features = split_features(reference_df)
feature_columns = numerical_features + categorical_features
del reference_df

TEST_CONFIGS = {
    "cicids2017": ["cicids2017", "iot23", "unsw_nb15"],
    "unsw_nb15": ["unsw_nb15", "cicids2017", "iot23"],
    "iot23": ["iot23", "cicids2017", "unsw_nb15"],
    "cicids2017_unsw_nb15": ["iot23"],
    "cicids2017_iot23": ["unsw_nb15"],
    "unsw_nb15_iot23": ["cicids2017"],
}

TEST_DATASETS = ["cicids2017", "iot23", "unsw_nb15"]
SINGLE_DATASET_TRAINING = set(TEST_DATASETS)
MODELS_TO_EVALUATE = [
    "decision_tree",
    "random_forest",
    "xgboost",
    "linear_svm",
    "mlp",
]
SELECTION_METHOD = "randomized_search"

In [12]:
CV_METRICS = [
    "precision_attack",
    "recall_attack",
    "f1_attack",
    "precision_benign",
    "recall_benign",
    "f1_benign",
    "f1_macro",
    "accuracy",
    "balanced_accuracy",
]


def load_training_summaries():
    frames = []

    for model_name in MODELS_TO_EVALUATE:
        summary_path = TRAINING_RESULTS_DIR / model_name / "summary.csv"
        if not summary_path.exists():
            raise FileNotFoundError(
                f"Resumo de treinamento não encontrado: {summary_path}. "
                "Execute primeiro o notebook 07_model_training.ipynb."
            )
        frames.append(pd.read_csv(summary_path))

    summaries = pd.concat(frames, ignore_index=True)
    return summaries.set_index(["model", "training_source"])


def scenario_type(training_name, test_name):
    if training_name == test_name:
        return "intra_dataset"
    if training_name in SINGLE_DATASET_TRAINING:
        return "cross_dataset"
    return "combined_to_unseen"


def training_metadata(summary_row):
    metadata = {
        "search_method": summary_row.get("search_method", SELECTION_METHOD),
        "selection_metric": summary_row.get("selection_metric"),
        "training_rows": summary_row.get("training_rows"),
        "params": summary_row.get("params", "{}"),
        "cv_n_splits": summary_row.get("n_splits"),
        "random_state": summary_row.get("random_state"),
        "training_elapsed_seconds": summary_row.get("elapsed_seconds"),
        "training_elapsed_minutes": summary_row.get("elapsed_minutes"),
        "final_fit_time_seconds": summary_row.get("refit_time_seconds"),
        "model_size_mb": summary_row.get("model_size_mb"),
    }

    for metric in CV_METRICS:
        metadata[f"cv_{metric}_mean"] = summary_row.get(f"{metric}_mean")
        metadata[f"cv_{metric}_std"] = summary_row.get(f"{metric}_std")

    return metadata

In [13]:
columns_to_load = feature_columns + ["label_binary"]
test_dataframes = {
    dataset_name: pd.read_parquet(
        DATA_DIR / f"{dataset_name}_test.parquet",
        columns=columns_to_load,
    )
    for dataset_name in TEST_DATASETS
}

training_summaries = load_training_summaries()
evaluation_results = []

for training_name, test_names in TEST_CONFIGS.items():
    for model_name in MODELS_TO_EVALUATE:
        model_path = MODEL_DIR / model_name / f"{training_name}.joblib"
        if not model_path.exists():
            raise FileNotFoundError(f"Modelo vencedor não encontrado: {model_path}")

        pipeline = joblib.load(model_path)
        summary_row = training_summaries.loc[(model_name, training_name)]
        report_dir = OUTPUT_DIR / training_name / model_name
        report_dir.mkdir(parents=True, exist_ok=True)

        for test_name in test_names:
            test_df = test_dataframes[test_name]
            X_test = test_df[feature_columns]
            y_test = test_df["label_binary"].eq("ATTACK").astype("int8")

            prediction_start = perf_counter()
            predictions = pipeline.predict(X_test)
            prediction_time_seconds = perf_counter() - prediction_start

            metrics_start = perf_counter()
            report_values = classification_report(
                y_test,
                predictions,
                labels=[0, 1],
                target_names=["BENIGN", "ATTACK"],
                output_dict=True,
                zero_division=0,
            )
            report_text = classification_report(
                y_test,
                predictions,
                labels=[0, 1],
                target_names=["BENIGN", "ATTACK"],
                digits=4,
                zero_division=0,
            )
            matrix_values = confusion_matrix(y_test, predictions, labels=[0, 1])
            balanced_accuracy = balanced_accuracy_score(y_test, predictions)
            metrics_calculation_time_seconds = perf_counter() - metrics_start

            matrix = pd.DataFrame(
                matrix_values,
                index=["Real BENIGN", "Real ATTACK"],
                columns=["Predito BENIGN", "Predito ATTACK"],
            )
            true_negative, false_positive, false_negative, true_positive = (
                matrix_values.ravel()
            )

            title = (
                f"Treino: {training_name} | Modelo: {model_name} | "
                f"Teste: {test_name}"
            )
            report_path = report_dir / f"randomized_search_{test_name}.txt"
            matrix_path = (
                report_dir / f"randomized_search_{test_name}_confusion_matrix.txt"
            )
            report_path.write_text(
                f"{title}\n\n{report_text}",
                encoding="utf-8",
            )
            matrix_path.write_text(
                f"{title}\n\n{matrix}",
                encoding="utf-8",
            )

            result = {
                "model": model_name,
                "selection_method": SELECTION_METHOD,
                "training_source": training_name,
                "test_source": test_name,
                "scenario_type": scenario_type(training_name, test_name),
                "test_rows": len(test_df),
                "benign_support": int(report_values["BENIGN"]["support"]),
                "attack_support": int(report_values["ATTACK"]["support"]),
                "accuracy": report_values["accuracy"],
                "balanced_accuracy": balanced_accuracy,
                "precision_benign": report_values["BENIGN"]["precision"],
                "recall_benign": report_values["BENIGN"]["recall"],
                "f1_benign": report_values["BENIGN"]["f1-score"],
                "precision_attack": report_values["ATTACK"]["precision"],
                "recall_attack": report_values["ATTACK"]["recall"],
                "f1_attack": report_values["ATTACK"]["f1-score"],
                "f1_macro": report_values["macro avg"]["f1-score"],
                "f1_weighted": report_values["weighted avg"]["f1-score"],
                "true_negative": int(true_negative),
                "false_positive": int(false_positive),
                "false_negative": int(false_negative),
                "true_positive": int(true_positive),
                "prediction_time_seconds": prediction_time_seconds,
                "prediction_rows_per_second": (
                    len(test_df) / prediction_time_seconds
                    if prediction_time_seconds > 0
                    else float("inf")
                ),
                "metrics_calculation_time_seconds": metrics_calculation_time_seconds,
                "model_file": str(model_path.relative_to(ROOT)),
                "classification_report_file": str(report_path.relative_to(ROOT)),
                "confusion_matrix_file": str(matrix_path.relative_to(ROOT)),
            }
            result.update(training_metadata(summary_row))
            evaluation_results.append(result)

evaluation_df = pd.DataFrame(evaluation_results).sort_values(
    ["training_source", "test_source", "model"]
)

expected_evaluations = len(MODELS_TO_EVALUATE) * sum(
    len(test_names) for test_names in TEST_CONFIGS.values()
)
assert len(evaluation_df) == expected_evaluations
assert not evaluation_df.duplicated(
    ["model", "training_source", "test_source"]
).any()

evaluation_df.to_csv(OUTPUT_DIR / "evaluation_summary.csv", index=False)
display(evaluation_df)

,model,selection_method,training_source,test_source,scenario_type,test_rows,benign_support,attack_support,accuracy,balanced_accuracy,...,cv_recall_benign_mean,cv_recall_benign_std,cv_f1_benign_mean,cv_f1_benign_std,cv_f1_macro_mean,cv_f1_macro_std,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_balanced_accuracy_std
0,decision_tree,randomized_search,cicids2017,cicids2017,intra_dataset,100925,80740,20185,0.994293,0.992216,...,0.995584,0.000227,0.996335,0.000194,0.990864,0.000483,0.994140,0.000310,0.991974,0.000494
9,linear_svm,randomized_search,cicids2017,cicids2017,intra_dataset,100925,80740,20185,0.921982,0.935038,...,0.914057,0.001798,0.949801,0.001052,0.890928,0.001984,0.922706,0.001556,0.935679,0.001265
12,mlp,randomized_search,cicids2017,cicids2017,intra_dataset,100925,80740,20185,0.994580,0.994179,...,0.994596,0.000255,0.996408,0.000174,0.991084,0.000433,0.994263,0.000279,0.993763,0.000553
3,random_forest,randomized_search,cicids2017,cicids2017,intra_dataset,100925,80740,20185,0.995690,0.996377,...,0.995132,0.000242,0.997141,0.000076,0.992910,0.000184,0.995435,0.000121,0.995889,0.000122
6,xgboost,randomized_search,cicids2017,cicids2017,intra_dataset,100925,80740,20185,0.995046,0.994414,...,0.995419,0.000104,0.996938,0.000088,0.992391,0.000220,0.995108,0.000141,0.994642,0.000373
1,decision_tree,randomized_search,cicids2017,iot23,cross_dataset,100925,80740,20185,0.093882,0.063581,...,0.995584,0.000227,0.996335,0.000194,0.990864,0.000483,0.994140,0.000310,0.991974,0.000494
10,linear_svm,randomized_search,cicids2017,iot23,cross_dataset,100925,80740,20185,0.125895,0.151604,...,0.914057,0.001798,0.949801,0.001052,0.890928,0.001984,0.922706,0.001556,0.935679,0.001265
13,mlp,randomized_search,cicids2017,iot23,cross_dataset,100925,80740,20185,0.787169,0.493894,...,0.994596,0.000255,0.996408,0.000174,0.991084,0.000433,0.994263,0.000279,0.993763,0.000553
4,random_forest,randomized_search,cicids2017,iot23,cross_dataset,100925,80740,20185,0.096111,0.062670,...,0.995132,0.000242,0.997141,0.000076,0.992910,0.000184,0.995435,0.000121,0.995889,0.000122
7,xgboost,randomized_search,cicids2017,iot23,cross_dataset,100925,80740,20185,0.094288,0.061549,...,0.995419,0.000104,0.996938,0.000088,0.992391,0.000220,0.995108,0.000141,0.994642,0.000373


In [14]:
evaluation_counts = (
    evaluation_df.groupby("scenario_type", as_index=False)
    .size()
    .rename(columns={"size": "evaluations"})
)
display(evaluation_counts)

,scenario_type,evaluations
0,combined_to_unseen,15
1,cross_dataset,30
2,intra_dataset,15


In [15]:
def show_reports(
    training_name,
    test_name,
    show_classification_report=True,
    show_confusion_matrix=True,
):
    for model_name in MODELS_TO_EVALUATE:
        report_dir = OUTPUT_DIR / training_name / model_name

        if show_classification_report:
            report_path = report_dir / f"randomized_search_{test_name}.txt"
            print(report_path.read_text(encoding="utf-8"))

        if show_confusion_matrix:
            matrix_path = (
                report_dir / f"randomized_search_{test_name}_confusion_matrix.txt"
            )
            print(matrix_path.read_text(encoding="utf-8"))

In [16]:
show_reports("cicids2017", "cicids2017")

Treino: cicids2017 | Modelo: decision_tree | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9972    0.9957    0.9964     80740
      ATTACK     0.9828    0.9888    0.9858     20185

    accuracy                         0.9943    100925
   macro avg     0.9900    0.9922    0.9911    100925
weighted avg     0.9943    0.9943    0.9943    100925

Treino: cicids2017 | Modelo: decision_tree | Teste: cicids2017

             Predito BENIGN  Predito ATTACK
Real BENIGN           80391             349
Real ATTACK             227           19958
Treino: cicids2017 | Modelo: random_forest | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9994    0.9952    0.9973     80740
      ATTACK     0.9812    0.9975    0.9893     20185

    accuracy                         0.9957    100925
   macro avg     0.9903    0.9964    0.9933    100925
weighted avg     0.9958    0.9957    0.9957    100925

Treino: cicids2017 | M

In [17]:
show_reports("cicids2017", "unsw_nb15")

Treino: cicids2017 | Modelo: decision_tree | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.7973    0.9799    0.8792     80740
      ATTACK     0.0402    0.0034    0.0062     20185

    accuracy                         0.7846    100925
   macro avg     0.4187    0.4916    0.4427    100925
weighted avg     0.6459    0.7846    0.7046    100925

Treino: cicids2017 | Modelo: decision_tree | Teste: unsw_nb15

             Predito BENIGN  Predito ATTACK
Real BENIGN           79117            1623
Real ATTACK           20117              68
Treino: cicids2017 | Modelo: random_forest | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.8000    0.9999    0.8888     80740
      ATTACK     0.0000    0.0000    0.0000     20185

    accuracy                         0.7999    100925
   macro avg     0.4000    0.5000    0.4444    100925
weighted avg     0.6400    0.7999    0.7111    100925

Treino: cicids2017 | Mode

In [18]:
show_reports("cicids2017_iot23", "unsw_nb15")

Treino: cicids2017_iot23 | Modelo: decision_tree | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.8055    0.9928    0.8894     80740
      ATTACK     0.5856    0.0408    0.0763     20185

    accuracy                         0.8024    100925
   macro avg     0.6955    0.5168    0.4828    100925
weighted avg     0.7615    0.8024    0.7268    100925

Treino: cicids2017_iot23 | Modelo: decision_tree | Teste: unsw_nb15

             Predito BENIGN  Predito ATTACK
Real BENIGN           80157             583
Real ATTACK           19361             824
Treino: cicids2017_iot23 | Modelo: random_forest | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.8044    0.9987    0.8911     80740
      ATTACK     0.8467    0.0287    0.0556     20185

    accuracy                         0.8047    100925
   macro avg     0.8256    0.5137    0.4733    100925
weighted avg     0.8129    0.8047    0.7240    100925

Treino:

In [19]:
show_reports("unsw_nb15", "unsw_nb15")

Treino: unsw_nb15 | Modelo: decision_tree | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.9987    0.9847    0.9917     80740
      ATTACK     0.9422    0.9950    0.9679     20185

    accuracy                         0.9868    100925
   macro avg     0.9705    0.9899    0.9798    100925
weighted avg     0.9874    0.9868    0.9869    100925

Treino: unsw_nb15 | Modelo: decision_tree | Teste: unsw_nb15

             Predito BENIGN  Predito ATTACK
Real BENIGN           79508            1232
Real ATTACK             100           20085
Treino: unsw_nb15 | Modelo: random_forest | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.9996    0.9846    0.9920     80740
      ATTACK     0.9418    0.9985    0.9693     20185

    accuracy                         0.9874    100925
   macro avg     0.9707    0.9915    0.9807    100925
weighted avg     0.9881    0.9874    0.9875    100925

Treino: unsw_nb15 | Modelo: 

In [20]:
show_reports("iot23", "iot23")

Treino: iot23 | Modelo: decision_tree | Teste: iot23

              precision    recall  f1-score   support

      BENIGN     0.9984    0.9999    0.9992     80740
      ATTACK     0.9995    0.9937    0.9966     20185

    accuracy                         0.9986    100925
   macro avg     0.9990    0.9968    0.9979    100925
weighted avg     0.9986    0.9986    0.9986    100925

Treino: iot23 | Modelo: decision_tree | Teste: iot23

             Predito BENIGN  Predito ATTACK
Real BENIGN           80730              10
Real ATTACK             127           20058
Treino: iot23 | Modelo: random_forest | Teste: iot23

              precision    recall  f1-score   support

      BENIGN     0.9985    0.9999    0.9992     80740
      ATTACK     0.9998    0.9939    0.9968     20185

    accuracy                         0.9987    100925
   macro avg     0.9991    0.9969    0.9980    100925
weighted avg     0.9987    0.9987    0.9987    100925

Treino: iot23 | Modelo: random_forest | Teste: iot23